In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path

import numpy as np
import healpy as hp

## General Setup

In [4]:
# Function for calculating opening angle
def find_opening_angle(true_zen, reco_zen, true_az, reco_az):
    return np.arccos(np.cos(reco_zen) * np.cos(true_zen) +
                     np.sin(reco_zen) * np.sin(true_zen) *
                     np.cos(reco_az - true_az))

# Function for weighted quantiles
def weighted_quantiles(values, weights, quantiles=[0.5]):
    if len(values) == 0:
        return 0
    else:
        i = np.argsort(values)
        c = np.cumsum(weights[i])
        return values[i[np.searchsorted(c, np.asarray(quantiles) * c[-1])]]

## Full-Sky

### Event Counts

In [5]:
data_dir = Path('/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/sidereal_unblinded')
tiers = ['tier1','tier2','tier3','tier4']

for tier in tiers:

    map_files = list(data_dir.glob(f'{tier}/reconstruction/relintensityiter/*.fits.gz'))
    sample_map = map_files[0]  # Just need the counts. Doesn't matter which iteration we pick

    data, _, _ = hp.read_map(sample_map, range(3))
    counts = data.sum()

    print(f'Tier {tier[-1]}:  {counts/1e8:.02f} x 10^8')

Tier 1:  5.52 x 10^8
Tier 2:  7.58 x 10^8
Tier 3:  2.51 x 10^8
Tier 4:  0.31 x 10^8


### Angular Resolution

In [5]:
year = 2012
model = 'SIBYLL2.1'

# Load in the SIBYLL2.1 particle sims - note that these will also be your variable names
KEYS = ['energy', 'hits', 'reco_pass', 'showerplane_zen', 'showerplane_az', 'laputop_zen', 'laputop_az', 'true_zenith', 'true_azimuth', 'Gweights']

for key in KEYS:
    sim_file = f'/data/user/tfutrell/it_anisotropy/{year}/{model}/{key}.npy'
    with open(sim_file, 'rb') as file:
        globals()[key] = np.load(file)

# Directional reconstruction cuts
plane_cut   = (showerplane_zen < np.radians(55))
laputop_cut = (laputop_zen < np.radians(55))
reco_cut    = (reco_pass == 1)

# Energy tier cuts
offset = np.ceil((year - 2011)/2)
TIERS = {
    'Tier 1': (3 <= hits) * (hits < 5),
    'Tier 2': (5 <= hits) * (hits < 10 - offset),
    'Tier 3': (10 - offset <= hits) * (hits < 17 - offset),
    'Tier 4': (17 - offset <= hits)
}

# Weights
it_weights = Gweights

In [14]:
for tier, tier_cut in TIERS.items():

    if tier == 'Tier 1':
        combined_cut = tier_cut * plane_cut
        d_psi = find_opening_angle(true_zenith, showerplane_zen, true_azimuth, showerplane_az)
    else:
        combined_cut = tier_cut * laputop_cut * reco_cut
        d_psi = find_opening_angle(true_zenith, laputop_zen, true_azimuth, laputop_az)
   
    # Find median energy for tier
    median_e, _, _ = weighted_quantiles(energy[combined_cut], it_weights[combined_cut], quantiles=[.5,.16,.84]) / 1e6

    # Find median opening angle for tier
    median_psi, sigl_psi, sigr_psi = weighted_quantiles(d_psi[combined_cut], it_weights[combined_cut], quantiles=[.5,.16,.84])

    print(f'{tier} ({median_e:.2f} PeV): {np.degrees(median_psi):.2f} -{np.degrees(median_psi-sigl_psi):.2f} +{np.degrees(sigr_psi-median_psi):.2f}')

Tier 1 (0.30 PeV): 1.74 -0.93 +1.96
Tier 2 (0.90 PeV): 0.72 -0.38 +0.66
Tier 3 (2.28 PeV): 0.48 -0.25 +0.39
Tier 4 (6.55 PeV): 0.33 -0.17 +0.27


## Split-Sky

In [15]:
it_split = 38.

### Event Counts

In [23]:
data_dir = Path('/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/sidereal_unblinded')
tiers = ['tier1','tier2','tier3','tier4']
count_dict = {}

for tier in tiers:

    map_files = list(data_dir.glob(f'{tier}/reconstruction/relintensityiter/*.fits.gz'))
    sample_map = map_files[0]  # Just need the counts. Doesn't matter which iteration we pick

    data, _, _ = hp.read_map(sample_map, range(3))
    counts = data.sum()

    npix = len(data)
    nside = hp.npix2nside(npix)
    theta, phi = hp.pix2ang(nside, range(npix))

    theta_min = [np.radians(180-it_split), np.radians(180-55)]
    theta_max = [np.pi, np.radians(180-it_split)]
    
    for j, (th_min, th_max) in enumerate(zip(theta_min, theta_max)):

        theta_cut = (theta >= th_min) * (theta < th_max)
        counts = data[theta_cut].sum()

        deg_range = f'{180 - np.degrees(th_max):.0f} - {180 - np.degrees(th_min):.0f}'
        
        print(f'Tier {tier[-1]} ({deg_range:>7}):  {counts/1e8:.02f} x 10^8')

        # Modify tier name
        tier_mod = 'a' if j==0 else 'b'
        count_dict[f'Tier {tier[-1]}{tier_mod}'] = counts

Tier 1 ( 0 - 38):  4.74 x 10^8
Tier 1 (38 - 55):  0.78 x 10^8
Tier 2 ( 0 - 38):  6.27 x 10^8
Tier 2 (38 - 55):  1.32 x 10^8
Tier 3 ( 0 - 38):  2.07 x 10^8
Tier 3 (38 - 55):  0.44 x 10^8
Tier 4 ( 0 - 38):  0.24 x 10^8
Tier 4 (38 - 55):  0.07 x 10^8


### Angular Resolution

In [18]:
for tier, tier_cut in TIERS.items():

    if tier == 'Tier 1':
        combined_cut = tier_cut * plane_cut
        d_psi = find_opening_angle(true_zenith, showerplane_zen, true_azimuth, showerplane_az)
        z_cut = (showerplane_zen <= np.radians(it_split))
    else:
        combined_cut = tier_cut * laputop_cut * reco_cut
        d_psi = find_opening_angle(true_zenith, laputop_zen, true_azimuth, laputop_az)
        z_cut = (laputop_zen <= np.radians(it_split))

    z_dict = {'lo':combined_cut * z_cut, 'hi':combined_cut * ~z_cut}
    for key, cut in z_dict.items():
   
        # Find median energy for tier
        median_e, _, _ = weighted_quantiles(energy[cut], it_weights[cut], quantiles=[.5,.16,.84]) / 1e6
    
        # Find median opening angle for tier
        median_psi, sigl_psi, sigr_psi = weighted_quantiles(d_psi[cut], it_weights[cut], quantiles=[.5,.16,.84])
    
        print(f'{tier} ({median_e:.2f} PeV) ({key}): {np.degrees(median_psi):.2f} -{np.degrees(median_psi-sigl_psi):.2f} +{np.degrees(sigr_psi-median_psi):.2f}')

Tier 1 (0.28 PeV) (lo): 1.72 -0.92 +1.89
Tier 1 (0.44 PeV) (hi): 1.87 -1.02 +2.32
Tier 2 (0.83 PeV) (lo): 0.71 -0.38 +0.65
Tier 2 (1.41 PeV) (hi): 0.75 -0.40 +0.71
Tier 3 (2.10 PeV) (lo): 0.47 -0.25 +0.38
Tier 3 (3.29 PeV) (hi): 0.51 -0.27 +0.42
Tier 4 (6.01 PeV) (lo): 0.33 -0.17 +0.26
Tier 4 (9.19 PeV) (hi): 0.36 -0.19 +0.31


### LaTeX Table Output

In [19]:
def round_and_format(num):
    # 1. Format to 2 sig figs (.2g)
    # 2. Convert back to float and format with commas and 0 decimals (,: .0f)
    return f'{float(f'{num:.2g}'):,.0f}'

In [33]:
# 1 & 300   & 5.57 & 2.58$^{+9.47}_{-1.62}$ \\[0.1cm]

for tier, tier_cut in TIERS.items():

    if tier == 'Tier 1':
        combined_cut = tier_cut * plane_cut
        d_psi = find_opening_angle(true_zenith, showerplane_zen, true_azimuth, showerplane_az)
        z_cut = (showerplane_zen <= np.radians(it_split))
    else:
        combined_cut = tier_cut * laputop_cut * reco_cut
        d_psi = find_opening_angle(true_zenith, laputop_zen, true_azimuth, laputop_az)
        z_cut = (laputop_zen <= np.radians(it_split))

    z_dict = {'lo':combined_cut * z_cut, 'hi':combined_cut * ~z_cut}
    for key, cut in z_dict.items():

        # Modify tier name
        tier_mod = 'a' if key=='lo' else 'b'
   
        # Find median energy for tier and format for printing
        median_e, _, _ = weighted_quantiles(energy[cut], it_weights[cut], quantiles=[.5,.16,.84])
        median_e_str = round_and_format(median_e / 1e3)
    
        # Find median opening angle for tier
        median_psi, sigl_psi, sigr_psi = weighted_quantiles(d_psi[cut], it_weights[cut], quantiles=[.5,.16,.84])

        # Count string
        count_str = f'{count_dict[tier+tier_mod]/1e8:.2f}'

        # Angular resolution string
        ang_res_str = f'{np.degrees(median_psi):.2f}$_{{-{np.degrees(median_psi-sigl_psi):.2f}}}^{{+{np.degrees(sigr_psi-median_psi):.2f}}}$'

        print(f'{tier[-1]}{tier_mod} & {median_e_str:>5} & {count_str} & {ang_res_str} \\\\[0.1cm]')

1a &   280 & 4.74 & 1.72$_{-0.92}^{+1.89}$ \\[0.1cm]
1b &   440 & 0.78 & 1.87$_{-1.02}^{+2.32}$ \\[0.1cm]
2a &   830 & 6.27 & 0.71$_{-0.38}^{+0.65}$ \\[0.1cm]
2b & 1,400 & 1.32 & 0.75$_{-0.40}^{+0.71}$ \\[0.1cm]
3a & 2,100 & 2.07 & 0.47$_{-0.25}^{+0.38}$ \\[0.1cm]
3b & 3,300 & 0.44 & 0.51$_{-0.27}^{+0.42}$ \\[0.1cm]
4a & 6,000 & 0.24 & 0.33$_{-0.17}^{+0.26}$ \\[0.1cm]
4b & 9,200 & 0.07 & 0.36$_{-0.19}^{+0.31}$ \\[0.1cm]
